In [1]:
import bffile
from pathlib import Path
import numpy as np
import tifffile as tifff
import ome_types as ot
from copy import deepcopy

In [6]:
a="lzw"
if a in (None,"lzw"):
    print("yeap")

yeap


In [14]:
dir(slice(0,21,3))

['__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 'indices',
 'start',
 'step',
 'stop']

In [30]:
from itertools import batched
no_of_series=21
batch_size=3


In [32]:
batches=[list(element) for element in batched(range(no_of_series),batch_size)]
for i in batches:
    print(i)

[0, 1, 2]
[3, 4, 5]
[6, 7, 8]
[9, 10, 11]
[12, 13, 14]
[15, 16, 17]
[18, 19, 20]


In [53]:
#img_path=Path(r"Y:\Ti2-W1\Kim Boonekamp\KB_2026_041_R3_myh9_gfp\raw_data\KB_2026_0408_07myh9-002_crop.nd2")
img_path=Path(r"Y:\Ti2-W1\Kim Boonekamp\KB_2026_055_MSI_H2B_GFP_live_R3\KB_2026_055_msi_H2B_GFP_R3.nd2")
#img_path=Path(r"C:\Users\VictorP\Downloads\KB_2026_0408_07myh9-001.nd2")

In [ ]:
bf=bffile.BioFile(img_path)
bf.open()
metadata={"series_count":None,"ome":None,"core":None}
metadata["series_count"]=bf.series_count()
metadata["ome"]=bf.ome_metadata
metadata["core"]=bf.core_metadata()

In [77]:
bf.close

<bound method BioFile.close of BioFile('KB_2026_055_msi_H2B_GFP_R3.nd2', 19 series)>

In [ ]:

with bffile.BioFile(img_path) as bf:
    metadata["series_count"]=bf.series_count()
    metadata["ome"]=bf.ome_metadata
    metadata["core"]=bf.core_metadata()

"""
metadata["series"]=[]
for i in range(metadata["series_count"]):
    metadata["series"].append({"id":i,
                               "x":metadata["ome"].images[i].pixels.planes[0].position_x,
                               "y":metadata["ome"].images[i].pixels.planes[0].position_y,
                               "channels":[ ch.name for ch in metadata["ome"].images[i].pixels.channels]
                               }
                               )
"""

'\nmetadata["series"]=[]\nfor i in range(metadata["series_count"]):\n    metadata["series"].append({"id":i,\n                               "x":metadata["ome"].images[i].pixels.planes[0].position_x,\n                               "y":metadata["ome"].images[i].pixels.planes[0].position_y,\n                               "channels":[ ch.name for ch in metadata["ome"].images[i].pixels.channels]\n                               }\n                               )\n'

In [64]:
metadata["ome"].images[0].pixels.channels

[Channel(
    id='Channel:0:0',
    name='FITC',
    detector_settings={'id': 'Detector:0:0'},
    light_path={},
    samples_per_pixel=1,
    emission_wavelength=525.0,
    color=Color('#24ff00', rgb=(36, 255, 0)),
 ),
 Channel(
    id='Channel:0:1',
    name='bf',
    detector_settings={'id': 'Detector:0:0'},
    light_path={},
    samples_per_pixel=1,
 )]

In [56]:
    bf=bffile.BioFile(img_path)
    bf.open()

    # Loop over xy series, i.e. rois

    lazy_array = bf.as_array(series=0,resolution=0)

In [57]:
bfield_map={ch.name:n for n,ch in enumerate(metadata["ome"].images[0].pixels.channels) 
                    if ch.name in ("bf","BF")
                    }
bf_channels=list(bfield_map.values())

In [70]:
def find_home_plane(lazy_array,ref_channel=0,ref_frame=0):
    # Sum along YX-axis (3,4)
    #result= lazy_array.to_dask().sum(axis=(3,4),keepdims=True).compute()
    channel_slice=slice(ref_channel,ref_channel+1)
    frame_slice=slice(ref_frame,ref_frame+1)
    aux_arr=np.array(lazy_array[frame_slice,channel_slice,:,:,:])
    result=np.sum(aux_arr,axis=(3,4),keepdims=True)
    result=(np.floor(result).astype("int32")).squeeze()
    z_home_index= int(np.argwhere(result)[0][0])
    return z_home_index


In [68]:
ref_channel=1
ref_frame=0
channel_slice=slice(ref_channel,ref_channel+1)
frame_slice=slice(ref_frame,ref_frame+1)
aux_arr=np.array(lazy_array[frame_slice,channel_slice,:,:,:])

In [73]:
result=np.sum(aux_arr,axis=(3,4),keepdims=True)
result=(np.floor(result).astype("int32")).squeeze()
z_home_index= int(np.argwhere(result)[0][0])
z_home_index

5

In [74]:
bf.close()

In [10]:
def remap_planes(planes_block,remap_dic,z_slice=None):
    edited_planes=[]
    for p in planes_block:
        if p.the_c in remap_dic:
            p.the_c=remap_dic[p.the_c]
            edited_planes.append(p)

    if isinstance(z_slice,int):
        for n in range(len(edited_planes)):
            edited_planes[n].the_z=0

    return edited_planes

def remap_channels(channels_block):
    edited_channels=[]
    for n,c in enumerate(channels_block):
        c.id=f"Channel:{0}:{n}"
        edited_channels.append(c)
    return edited_channels

def remap_images(images_block):
    edited_images=[]
    for n,im in enumerate(images_block):
        im.id=f"Image:{n}"
        edited_images.append(im)
    return edited_images

In [32]:
import ome_types as otm
channels_id=[0]
series_id=0

# Conform channels
channels_id=sorted(set(channels_id))
remap_ch_id = {old: new for new, old in enumerate(channels_id)}

# Initialize the ome-metadata for the series
ome_series=otm.model.OME()
ome_series.instruments=deepcopy(metadata["ome"].instruments)
#ome_series.images=[deepcopy(metadata["ome"].images[series_id])]
images_block=[deepcopy(metadata["ome"].images[series_id])]

# Edit images block
ome_series.images=remap_images(images_block)

# Edit channels block
channels_block=[ome_series.images[series_id].pixels.channels[ch] for ch in channels_id ]

ome_series.images[series_id].pixels.channels=remap_channels(channels_block)

# Edit the size_c
ome_series.images[series_id].pixels.size_c=len(channels_block)

# Edit the DimensionOrder
ome_series.images[series_id].pixels.dimension_order="XYZCT"

# Edit planes block
planes_unsorted=[ (plane,plane.the_t,plane.the_c,plane.the_z) 
                 for plane in ome_series.images[series_id].pixels.planes 
                 if np.isin(plane.the_c,channels_id) 
                 ]


planes_block = [ p[0] for p in sorted(planes_unsorted,key=lambda p: (p[1], p[2], p[3])) ]    
ome_series.images[series_id].pixels.planes=remap_planes(planes_block,remap_ch_id)

# Create TiffData block to warranty compatibility with visualizers
tiff_data=[ot.model.TiffData(first_t=p.the_t,first_c=p.the_c,first_z=p.the_z,ifd=n,plane_count=1) 
           for n,p in enumerate(planes_block)
           ]

ome_series.images[series_id].pixels.tiff_data_blocks=tiff_data

In [ ]:
def create_series_ome(src_metadata,channels,series_id,z_slice=None):

    # Conform channels
    channels_id=sorted(set(channels))
    remap_ch_id = {old: new for new, old in enumerate(channels_id)}

    # Initialize the ome-metadata for the series
    ome_series=ot.model.OME()
    ome_series.instruments=deepcopy(src_metadata.instruments)
    images_block=[deepcopy(src_metadata.images[series_id])]

    # Edit images block
    ome_series.images=remap_images(images_block)

    # Edit channels block
    channels_block=[ome_series.images[0].pixels.channels[ch] for ch in channels_id ]

    ome_series.images[0].pixels.channels=remap_channels(channels_block)

    # Edit the size_c
    ome_series.images[0].pixels.size_c=len(channels_block)

    # Edit the DimensionOrder
    ome_series.images[0].pixels.dimension_order="XYZCT"

    # Edit planes block
    #  Search channels
    planes_unsorted=[ (plane,plane.the_t,plane.the_c,plane.the_z) 
                 for plane in ome_series.images[0].pixels.planes 
                 if np.isin(plane.the_c,channels_id) 
                 ]
    
    planes_block = [ p[0] for p in sorted(planes_unsorted,key=lambda p: (p[1], p[2], p[3])) ]
    if isinstance(z_slice,int):
        planes_block=[p for p in planes_block if p.the_z==z_slice]
        ome_series.images[0].pixels.size_z=1
        
    ome_series.images[0].pixels.planes=remap_planes(planes_block,remap_ch_id,z_slice=z_slice)

    # Create TiffData block to warranty compatibility with visualizers
    tiff_data=[ot.model.TiffData(first_t=p.the_t,first_c=p.the_c,first_z=p.the_z,ifd=n,plane_count=1) 
           for n,p in enumerate(planes_block)
           ]

    ome_series.images[0].pixels.tiff_data_blocks=tiff_data

    return ome_series

In [164]:
ome_extra=create_series_ome(metadata["ome"],[1],1)

In [172]:
[p for p in ome_extra.images[0].pixels.planes if p.the_z==15]

[Plane(
    the_z=15,
    the_t=0,
    the_c=0,
    delta_t=39.85396202499978,
    exposure_time=0.1,
    position_x=-10826.1,
    position_x_unit='µm',
    position_y=-6374.5,
    position_y_unit='µm',
    position_z=4968.56,
    position_z_unit='µm',
 )]

In [ ]:
[p for p in ome_extra.images[0].pixels.planes if p.the_z==15]